# v13_2 Benchmark Runner — vast.ai RTX Pro 6000

Runs **v13_2 vs v13_1** head-to-head on ls20 (1200s/level) and ar25 (600s/level).
All BFS, pure CPU workers, fork context. Run cells top to bottom once.


In [ ]:
# 1. Clone the repo (skip if already cloned)
import os
REPO = "/root/arc3"
if not os.path.exists(REPO):
    !git clone https://github.com/shreyasmahimkar/arc-agi-3 {REPO}
else:
    !git -C {REPO} pull --ff-only
print("Repo ready at", REPO)


In [ ]:
# 2. Install arcengine + dependencies from bundled wheels
WHEELS = "/root/arc3/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"
!pip install --quiet --no-index --find-links {WHEELS} \n    arc-agi pydantic python-dotenv
print("Dependencies installed")


In [ ]:
# 3. Verify hardware — expect many cores, fork context
import multiprocessing, platform, subprocess
ncpu = multiprocessing.cpu_count()
print(f"CPUs: {ncpu}  |  OS: {platform.system()}  |  Workers we will use: {ncpu - 1}")
try:
    r = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                       capture_output=True, text=True, timeout=5)
    print("GPU:", r.stdout.strip())
except Exception:
    print("No nvidia-smi (CPU-only run is fine)")


## ls20 — 1200s/level
L5 needs this headroom. Expect ~2-3 hrs total for 7 levels × 2 versions.

In [ ]:
import os, multiprocessing
REPO = "/root/arc3"
SOLVER = os.path.join(REPO, "CommunitySolutions/chronos_solver/v13_2")
WORKERS = multiprocessing.cpu_count() - 1
os.makedirs(SOLVER, exist_ok=True)

# Run benchmark: v13_2 first, then v13_1
!cd {SOLVER} && PYTHONUNBUFFERED=1 python benchmark.py \n    --games ls20:7 \n    --versions .,../v13_1 \n    --budget 1200 \n    --workers {WORKERS} \n    --max-states 10000000 \n    --out ls20_benchmark_1200s.json


## ar25 — 600s/level
Space-limited, not time-limited — 600s is enough.

In [ ]:
import os, multiprocessing
REPO = "/root/arc3"
SOLVER = os.path.join(REPO, "CommunitySolutions/chronos_solver/v13_2")
WORKERS = multiprocessing.cpu_count() - 1

!cd {SOLVER} && PYTHONUNBUFFERED=1 python benchmark.py \n    --games ar25:3 \n    --versions .,../v13_1 \n    --budget 600 \n    --workers {WORKERS} \n    --max-states 10000000 \n    --out ar25_benchmark_600s.json


## Results

In [ ]:
import json, os
SOLVER = "/root/arc3/CommunitySolutions/chronos_solver/v13_2"

for fname in ["ls20_benchmark_1200s.json", "ar25_benchmark_600s.json"]:
    path = os.path.join(SOLVER, fname)
    if not os.path.exists(path):
        print(f"{fname}: not found yet")
        continue
    rows = json.load(open(path))
    print(f"
=== {fname} ===")
    print(f"  {'Game':<8} {'Level':<6} {'Version':<10} {'Solved':<8} {'Actions':<10} {'Time(s)':<10}")
    for r in rows:
        solved = "YES" if r.get("solved") else "no"
        acts = r.get("actions", "-")
        t = round(r.get("elapsed", 0), 1)
        print(f"  {r['game']:<8} L{r['level']:<5} {r['version']:<10} {solved:<8} {str(acts):<10} {t}")


In [ ]:
# Print the markdown summary table written by benchmark.py
import os
SOLVER = "/root/arc3/CommunitySolutions/chronos_solver/v13_2"
for fname in ["BENCHMARK.md"]:
    p = os.path.join(SOLVER, fname)
    if os.path.exists(p):
        print(open(p).read())
